# First shot map: Argentina in the 2022 World Cup final

Goal: load one match from StatsBomb open data, keep one team's shots, and draw them on a half pitch.
Along the way we hit the most common silent mistake in this data: counting penalty shootout kicks as shots.

Data: StatsBomb open data. Terms and attribution rules are in `docs/DATA_SOURCES.md`.
Before committing, clear all outputs (notebook toolbar, "Clear All Outputs").

In [ ]:
import warnings

from mplsoccer import VerticalPitch
from statsbombpy import sb

warnings.filterwarnings("ignore")  # no-credentials warning is fine for open data

## 1. Find the match

StatsBomb organises the open data as competitions, then seasons, then matches.
`sb.competitions()` lists every competition and season with its IDs; the 2022 FIFA World Cup is
competition 43, season 106. `sb.matches()` returns one row per match, and `competition_stage`
tells us which round it was, so we keep the final.

In [ ]:
matches = sb.matches(competition_id=43, season_id=106)
final = matches[matches.competition_stage == "Final"].iloc[0]
print(final.match_id, final.home_team, final.home_score, "-", final.away_score, final.away_team)

## 2. Load the events and keep Argentina's shots

`sb.events()` returns one row per on-the-ball action (passes, carries, pressures, shots...).
The `type` column says what kind of action it is and `team` who made it.
Shot-specific details live in columns that start with `shot_`, for example `shot_outcome`
and `shot_statsbomb_xg` (StatsBomb's probability that the shot becomes a goal).

In [ ]:
events = sb.events(match_id=final.match_id)
shots = events[(events.type == "Shot") & (events.team == "Argentina")]
print(len(events), "events in the match,", len(shots), "of them Argentina shots")

## 3. The pitfall: period 5 is the penalty shootout

`period` is 1 and 2 for the halves, 3 and 4 for extra time, and 5 for the penalty shootout.
The final went to penalties, so shootout kicks appear as `Shot` events too.
Shootout kicks are not part of the match, so match stats must exclude period 5.
Nothing crashes if we forget: the goal count is just silently wrong. Compare the two counts below.

In [ ]:
print(shots.groupby("period").size())

goals_with_shootout = (shots.shot_outcome == "Goal").sum()
shots = shots[shots.period < 5]
goals_in_match = (shots.shot_outcome == "Goal").sum()
print("Goals if the shootout is left in:", goals_with_shootout)
print("Goals in the match (shootout excluded):", goals_in_match)

## 4. Draw the shot map

`VerticalPitch(pitch_type="statsbomb")` understands StatsBomb's own coordinates (120 x 80,
origin top-left), so we plot `location` as it comes. The pipeline converts to PitchLens internal
metres instead (see `docs/adr/0001-pitch-coordinates.md`); here we only draw.

Each circle is one shot. Its area is proportional to StatsBomb xG (the multiplier only makes the
circles visible). Goals are filled; every other outcome is an empty circle.

In [ ]:
pitch = VerticalPitch(pitch_type="statsbomb", half=True)
fig, ax = pitch.draw(figsize=(7, 6))
x, y = shots.location.str[0], shots.location.str[1]
goal = shots.shot_outcome == "Goal"
size = shots.shot_statsbomb_xg * 900
pitch.scatter(x[~goal], y[~goal], s=size[~goal], ax=ax, facecolors="none", edgecolors="black")
pitch.scatter(x[goal], y[goal], s=size[goal], ax=ax, color="#E8473B", edgecolors="black")
ax.set_title(
    "Argentina shots, 2022 World Cup final (excl. shootout)\n"
    "Circle size = StatsBomb xG. Data: StatsBomb open data",
    fontsize=10,
);